In [15]:
import json
import zipfile
from pathlib import Path

zip_path = r"C:\Users\Gargi\Downloads\COde-Dataset.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    # List everything inside to confirm exact paths
    names = z.namelist()
    json_files = [n for n in names if n.endswith('.json')]
    print("JSON files found:")
    for jf in json_files:
        print(" -", jf)
    
    # Peek into train.json structure
    with z.open('train.json') as f:
        data = json.load(f)
        print("\nTop-level keys in train.json:", list(data.keys()))
        
        if 'images' in data:
            print("\nSample image entries:")
            for img in data['images'][:5]:
                print(img)
        
        if 'annotations' in data:
            print("\nSample annotation entry:")
            print(data['annotations'][0])

JSON files found:
 - test_cls.json
 - test_diagnostic.json
 - train.json
 - train-cls.json
 - train-diagnostic.json


AttributeError: 'list' object has no attribute 'keys'

In [3]:
import json
import zipfile

zip_path = r"C:\Users\Gargi\Downloads\COde-Dataset.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open('train.json') as f:
        data = json.load(f)
        
        print("Type:", type(data))
        print("Length:", len(data))
        
        print("\nFirst entry:")
        print(data[0])
        
        print("\nType of first entry:", type(data[0]))
        
        # If entries are dicts, show their keys
        if isinstance(data[0], dict):
            print("\nKeys in first entry:", list(data[0].keys()))
        
        print("\nA few more entries for pattern-checking:")
        for entry in data[1:5]:
            print(entry)

Type: <class 'list'>
Length: 24497

First entry:
{'messages': [{'content': '根据给定的信息生成检查报告。\n\n图片:\n图片-1: <image>\n图片-2: <image>\n图片-3: <image>\n图片-4: <image>\n图片-5: <image>\n图片-6: <image>\n图片-7: <image>\n图片-8: <image>\n\nX光:\nX片-1: <image>\nX片-2: <image>\n\n患者信息:\n年龄: 11\n性别: 男\n主诉: 矫正14个月，复查\n现病史: 上下颌排齐中', 'role': 'user'}, {'content': '病历: 复诊病历|诊断：错合畸形安氏Ⅱ类(K07.200x012)。\n检查: 前牙深覆合覆盖\nX线检查示: \n诊断: 错合畸形安氏Ⅱ类(K07.200x012)\n治疗方案: \n治疗建议: \n处理: 给予患者最后—副牙套，拍照+口扫+拍片，微调。\n医嘱: 嘱嘴里一副牙套一直戴，不坏不换最后—副牙套。\n备注: ', 'role': 'assistant'}], 'images': ['Images/Photographs/3168-001-04.jpg', 'Images/Photographs/3168-001-07.jpg', 'Images/Photographs/3168-001-02.jpg', 'Images/Photographs/3168-001-06.jpg', 'Images/Photographs/3168-001-08.jpg', 'Images/Photographs/3168-001-01.jpg', 'Images/Photographs/3168-001-05.jpg', 'Images/Photographs/3168-001-03.jpg', 'Images/Radiographs/3168-001-02.jpg', 'Images/Radiographs/3168-001-01.jpg']}

Type of first entry: <class 'dict'>

Keys in first entry: ['messages', 'images']

In [4]:
import json
import re
import zipfile

zip_path = r"C:\Users\Gargi\Downloads\COde-Dataset.zip"

age_pattern_en = re.compile(r'Age:\s*(\d+)')
age_pattern_cn = re.compile(r'年龄:\s*(\d+)')

def extract_age(user_content):
    m = age_pattern_en.search(user_content)
    if m:
        return int(m.group(1))
    m = age_pattern_cn.search(user_content)
    if m:
        return int(m.group(1))
    return None

def process_json_file(z, json_name):
    records = []
    with z.open(json_name) as f:
        data = json.load(f)
    
    for entry in data:
        user_msg = next((m['content'] for m in entry['messages'] if m['role'] == 'user'), '')
        age = extract_age(user_msg)
        
        radiograph_files = [img for img in entry['images'] if img.startswith('Images/Radiographs/')]
        
        if age is not None and radiograph_files:
            for rf in radiograph_files:
                records.append({
                    'file_name': rf.split('/')[-1],  # just the filename, matches your folder
                    'full_path': rf,
                    'age': age,
                    'source_json': json_name
                })
    return records

all_records = []
with zipfile.ZipFile(zip_path, 'r') as z:
    for jf in ['train.json', 'test_cls.json', 'test_diagnostic.json', 'train-cls.json', 'train-diagnostic.json']:
        try:
            recs = process_json_file(z, jf)
            print(f"{jf}: {len(recs)} radiograph records with age found")
            all_records.extend(recs)
        except KeyError:
            print(f"{jf} not found in zip, skipping")

import pandas as pd
df = pd.DataFrame(all_records)
df = df.drop_duplicates(subset='file_name')  # in case same image appears in multiple json files
print(f"\nTotal unique radiograph-age records: {len(df)}")
df.to_csv(r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\extracted_age_from_json.csv", index=False)

train.json: 21606 radiograph records with age found
test_cls.json: 1351 radiograph records with age found
test_diagnostic.json: 1351 radiograph records with age found
train-cls.json: 13363 radiograph records with age found
train-diagnostic.json: 8243 radiograph records with age found

Total unique radiograph-age records: 7665


In [5]:
import pandas as pd
from pathlib import Path

extracted_csv = r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\extracted_age_from_json.csv"
existing_csv = r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\image_age.csv"
radiographs_dir = r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\Radiographs"

df_new = pd.read_csv(extracted_csv)
df_old = pd.read_csv(existing_csv)

new_files = set(df_new['file_name'])
old_files = set(df_old['Image_File'].astype(str))
folder_files = set(f.name for f in Path(radiographs_dir).iterdir() if f.is_file())

print(f"Unique files in extracted JSON data: {len(new_files)}")
print(f"Unique files in existing image_age.csv: {len(old_files)}")
print(f"Files currently in Radiographs folder: {len(folder_files)}")

print(f"\nOverlap between JSON data and existing CSV: {len(new_files & old_files)}")
print(f"Overlap between JSON data and current folder: {len(new_files & folder_files)}")
print(f"In JSON but not in current folder at all: {len(new_files - folder_files)}")
print(f"In current folder but not in JSON (no age available): {len(folder_files - new_files)}")

Unique files in extracted JSON data: 7665
Unique files in existing image_age.csv: 3860
Files currently in Radiographs folder: 880

Overlap between JSON data and existing CSV: 3860
Overlap between JSON data and current folder: 880
In JSON but not in current folder at all: 6785
In current folder but not in JSON (no age available): 0


In [6]:
import zipfile
import pandas as pd
from pathlib import Path

zip_path = r"C:\Users\Gargi\Downloads\COde-Dataset.zip"
extracted_csv = r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\extracted_age_from_json.csv"
radiographs_dir = Path(r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\Radiographs")

df_new = pd.read_csv(extracted_csv)

# Files already present locally
existing_files = set(f.name for f in radiographs_dir.iterdir() if f.is_file())

# Figure out which ones still need extracting
to_extract = df_new[~df_new['file_name'].isin(existing_files)]
print(f"Need to extract {len(to_extract)} images from the zip")

with zipfile.ZipFile(zip_path, 'r') as z:
    zip_namelist = set(z.namelist())
    
    extracted_count = 0
    missing_in_zip = []
    
    for _, row in to_extract.iterrows():
        full_path = row['full_path']  # e.g. Images/Radiographs/3168-001-02.jpg
        
        if full_path not in zip_namelist:
            missing_in_zip.append(full_path)
            continue
        
        # Read bytes from zip and write directly to Radiographs folder with just the filename
        with z.open(full_path) as src, open(radiographs_dir / row['file_name'], 'wb') as dst:
            dst.write(src.read())
        extracted_count += 1

print(f"\nExtracted {extracted_count} new images into {radiographs_dir}")
print(f"Missing from zip entirely: {len(missing_in_zip)}")
if missing_in_zip:
    print(missing_in_zip[:10])

Need to extract 6785 images from the zip

Extracted 6785 new images into C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\Radiographs
Missing from zip entirely: 0


In [9]:
import pandas as pd

extracted_csv = r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\extracted_age_from_json.csv"
existing_csv = r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\image_age.csv"

df_new = pd.read_csv(extracted_csv)
df_old = pd.read_csv(existing_csv)

df_old_renamed = df_old.rename(columns={'Image_File': 'file_name', 'Age': 'age'})

merged = pd.merge(
    df_old_renamed[['file_name', 'age']],
    df_new[['file_name', 'age']],
    on='file_name',
    suffixes=('_old', '_new')
)

mismatches = merged[merged['age_old'] != merged['age_new']]

print(f"Total overlapping records checked: {len(merged)}")
print(f"Mismatched ages: {len(mismatches)}")

if len(mismatches) > 0:
    print("\nSample mismatches:")
    print(mismatches.head(10))
else:
    print("\nAll ages match perfectly between old and new CSVs.")

Total overlapping records checked: 3860
Mismatched ages: 0

All ages match perfectly between old and new CSVs.


In [8]:
import pandas as pd

existing_csv = r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\image_age.csv"
df_old = pd.read_csv(existing_csv)

print("Columns in image_age.csv:", list(df_old.columns))
print("\nFirst few rows:")
print(df_old.head())


Columns in image_age.csv: ['Image_File', 'Age']

First few rows:
        Image_File  Age
0  4781-001-02.jpg   15
1  4588-001-01.jpg   10
2  4780-001-02.jpg   11
3  4780-001-01.jpg   11
4  4770-001-01.jpg   58


In [13]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import shutil

def is_collage(image_path, threshold=0.15):
    """
    Detects collages by looking for significant gaps in pixel intensity 
    projections that span the entire image.
    """
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    
    # Resize to speed up processing
    img_small = cv2.resize(img, (500, 500))
    _, thresh = cv2.threshold(img_small, 30, 255, cv2.THRESH_BINARY)
    
    # Calculate vertical and horizontal projections
    row_projection = np.sum(thresh, axis=1) / (thresh.shape[1] * 255)
    col_projection = np.sum(thresh, axis=0) / (thresh.shape[0] * 255)
    
    # Count how many times the projection falls below a 'gap' threshold
    # A single panoramic X-ray should be mostly continuous (few gaps)
    # A collage will have multiple clear gaps
    row_gaps = np.sum(row_projection < threshold)
    col_gaps = np.sum(col_projection < threshold)
    
    # If we have multiple distinct regions separated by gaps, flag as collage
    return row_gaps > 5 or col_gaps > 5

def filter_dataset(csv_path, img_dir, output_path, rejected_dir):
    df = pd.read_csv(csv_path)
    valid_data = []
    
    # Ensure the rejected folder exists
    Path(rejected_dir).mkdir(parents=True, exist_ok=True)
    
    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_path = Path(img_dir) / row['file_name']
        
        # If it's not a collage, keep it
        if not is_collage(img_path):
            valid_data.append(row)
        else:
            # Move rejected file into the rejected folder for cross-checking
            dest_path = Path(rejected_dir) / img_path.name
            if img_path.exists():
                shutil.move(str(img_path), str(dest_path))
            
    pd.DataFrame(valid_data).to_csv(output_path, index=False)
    print(f"Filtered dataset saved. Kept {len(valid_data)} images.")

# Usage
filter_dataset(
    r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\extracted_age_from_json.csv",
    r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\Radiographs",
    r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\cleaned_metadata.csv",
    r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\Rejected"
)

100%|█████████████████████████████████████████████████████████████████████████████| 7665/7665 [00:59<00:00, 127.85it/s]


Filtered dataset saved. Kept 4229 images.


In [14]:
import shutil
from pathlib import Path
#movinf files from rejected back to radiographs folder
rejected_dir = Path(r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\Rejected")
radiographs_dir = Path(r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\Radiographs")

restored = 0
for f in rejected_dir.iterdir():
    if f.is_file():
        shutil.move(str(f), str(radiographs_dir / f.name))
        restored += 1

print(f"Restored {restored} files back to Radiographs")

Restored 3436 files back to Radiographs


In [16]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

radiographs_dir = Path(r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\Radiographs")

def get_metrics(image_path):
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    
    h, w = img.shape
    aspect_ratio = w / h
    
    img_small = cv2.resize(img, (500, 250))
    left_half = img_small[:, :250]
    right_half = img_small[:, 250:]
    right_half_flipped = cv2.flip(right_half, 1)
    
    left_norm = cv2.equalizeHist(left_half)
    right_norm = cv2.equalizeHist(right_half_flipped)
    
    diff = cv2.absdiff(left_norm, right_norm)
    symmetry_score = np.mean(diff) / 255.0
    
    return {
        'file_name': image_path.name,
        'width': w,
        'height': h,
        'aspect_ratio': round(aspect_ratio, 3),
        'symmetry_score': round(symmetry_score, 3)
    }

# Sample 100 images
all_files = [f for f in radiographs_dir.iterdir() if f.is_file()]
sample_files = all_files[:100]  # first 100; use random.sample(all_files, 100) for a random sample instead

results = []
for f in sample_files:
    m = get_metrics(f)
    if m:
        results.append(m)

df_metrics = pd.DataFrame(results)

print("Aspect ratio stats:")
print(df_metrics['aspect_ratio'].describe())

print("\nSymmetry score stats:")
print(df_metrics['symmetry_score'].describe())

# Save for manual inspection
df_metrics.to_csv(r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\sample_metrics.csv", index=False)
print("\nSaved full details to sample_metrics.csv")

# Show the 10 most extreme cases on each metric — good candidates to manually check
print("\n--- Lowest aspect ratios (likely non-panoramic) ---")
print(df_metrics.nsmallest(10, 'aspect_ratio')[['file_name', 'aspect_ratio']])

print("\n--- Highest aspect ratios ---")
print(df_metrics.nlargest(10, 'aspect_ratio')[['file_name', 'aspect_ratio']])

print("\n--- Highest symmetry scores (most asymmetric) ---")
print(df_metrics.nlargest(10, 'symmetry_score')[['file_name', 'symmetry_score']])

Aspect ratio stats:
count    100.000000
mean       1.578240
std        0.518569
min        0.926000
25%        0.929000
50%        1.629000
75%        2.123000
max        2.207000
Name: aspect_ratio, dtype: float64

Symmetry score stats:
count    100.000000
mean       0.176530
std        0.075886
min        0.068000
25%        0.095000
50%        0.176500
75%        0.248250
max        0.334000
Name: symmetry_score, dtype: float64

Saved full details to sample_metrics.csv

--- Lowest aspect ratios (likely non-panoramic) ---
          file_name  aspect_ratio
10  0147-001-01.jpg         0.926
30  0152-004-02.jpg         0.926
51  0158-001-02.jpg         0.926
54  0158-002-02.jpg         0.926
78  0167-001-01.jpg         0.926
81  0168-001-02.jpg         0.926
94  0173-001-01.jpg         0.926
96  0173-002-01.jpg         0.926
0   0145-001-01.jpg         0.929
2   0145-002-01.jpg         0.929

--- Highest aspect ratios ---
          file_name  aspect_ratio
99  0174-001-02.jpg         2.2

In [17]:
import cv2
import numpy as np
import pandas as pd
import shutil
from pathlib import Path
from tqdm import tqdm

def is_collage(image_path, threshold=0.15):
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    
    img_small = cv2.resize(img, (500, 500))
    _, thresh = cv2.threshold(img_small, 30, 255, cv2.THRESH_BINARY)
    
    row_projection = np.sum(thresh, axis=1) / (thresh.shape[1] * 255)
    col_projection = np.sum(thresh, axis=0) / (thresh.shape[0] * 255)
    
    row_gaps = np.sum(row_projection < threshold)
    col_gaps = np.sum(col_projection < threshold)
    
    return row_gaps > 5 or col_gaps > 5

def get_aspect_ratio(image_path):
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    h, w = img.shape
    return w / h

def get_symmetry_score(image_path):
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    
    img_small = cv2.resize(img, (500, 250))
    left_half = img_small[:, :250]
    right_half = img_small[:, 250:]
    right_half_flipped = cv2.flip(right_half, 1)
    
    left_norm = cv2.equalizeHist(left_half)
    right_norm = cv2.equalizeHist(right_half_flipped)
    
    diff = cv2.absdiff(left_norm, right_norm)
    return np.mean(diff) / 255.0

def is_valid_radiograph(image_path, ar_min=1.8, ar_max=2.4, symmetry_threshold=0.35):
    if is_collage(image_path):
        return False, "collage"
    
    ar = get_aspect_ratio(image_path)
    if ar is None:
        return False, "unreadable"
    
    if not (ar_min <= ar <= ar_max):
        return False, f"bad_aspect_ratio_{ar:.2f}"
    
    symmetry = get_symmetry_score(image_path)
    if symmetry is not None and symmetry > symmetry_threshold:
        return False, f"asymmetric_{symmetry:.2f}"
    
    return True, "ok"

def filter_dataset(csv_path, img_dir, output_path, rejected_dir):
    df = pd.read_csv(csv_path)
    valid_data = []
    rejection_log = []
    
    Path(rejected_dir).mkdir(parents=True, exist_ok=True)
    
    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_path = Path(img_dir) / row['file_name']
        
        valid, reason = is_valid_radiograph(img_path)
        
        if valid:
            valid_data.append(row)
        else:
            dest_path = Path(rejected_dir) / img_path.name
            if img_path.exists():
                shutil.move(str(img_path), str(dest_path))
            rejection_log.append({'file_name': row['file_name'], 'reason': reason})
            
    pd.DataFrame(valid_data).to_csv(output_path, index=False)
    
    if rejection_log:
        rejection_log_path = Path(rejected_dir) / "rejection_log.csv"
        pd.DataFrame(rejection_log).to_csv(rejection_log_path, index=False)
        print(f"Rejection log saved to {rejection_log_path}")
    
    print(f"Filtered dataset saved. Kept {len(valid_data)} images. Rejected {len(rejection_log)}.")

# Usage
filter_dataset(
    r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\extracted_age_from_json.csv",
    r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\Radiographs",
    r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\cleaned_metadata.csv",
    r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\Rejected"
)

100%|█████████████████████████████████████████████████████████████████████████████| 7665/7665 [00:14<00:00, 541.93it/s]

Rejection log saved to C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\Rejected\rejection_log.csv
Filtered dataset saved. Kept 2078 images. Rejected 5587.


In [2]:
import pandas as pd
from pathlib import Path

radiographs_dir = Path(r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\Radiographs")
cleaned_csv = r"C:\Users\Gargi\Desktop\AgePrediction_with_DentalXray\data\image_age.csv"

def check_match(csv_path, label):
    df = pd.read_csv(csv_path)
    csv_files = set(df['file_name'].astype(str))
    folder_files = set(f.name for f in radiographs_dir.iterdir() if f.is_file())
    
    print(f"--- Checking against {label} ---")
    print(f"Rows in CSV: {len(df)}")
    print(f"Unique filenames in CSV: {len(csv_files)}")
    print(f"Files in Radiographs folder: {len(folder_files)}")
    
    duplicates = df['file_name'][df['file_name'].duplicated()]
    print(f"Duplicate filenames in CSV: {len(duplicates)}")
    
    in_csv_not_folder = csv_files - folder_files
    in_folder_not_csv = folder_files - csv_files
    
    print(f"In CSV but missing from folder: {len(in_csv_not_folder)}")
    if in_csv_not_folder:
        print("  Sample:", list(in_csv_not_folder)[:10])
    
    print(f"In folder but not in CSV: {len(in_folder_not_csv)}")
    if in_folder_not_csv:
        print("  Sample:", list(in_folder_not_csv)[:10])
    
    print(f"Perfect match: {csv_files == folder_files}\n")

check_match(cleaned_csv, "image_age.csv (post-filter)")

--- Checking against image_age.csv (post-filter) ---
Rows in CSV: 2078
Unique filenames in CSV: 2078
Files in Radiographs folder: 2078
Duplicate filenames in CSV: 0
In CSV but missing from folder: 0
In folder but not in CSV: 0
Perfect match: True

